<a href="https://colab.research.google.com/github/Coobaltoo13/milling-tool-life-fusion/blob/main/notebooks/02_mlp_lstm_sin_autoencoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Exploración Inicial - Milling Process Dataset
Fusión Multimodal de Vibración y Corriente para Predicción de Vida Útil en Fresado
Objetivo: Cargar el dataset y verificar su estructura inicial

1. Carga de datos
El dataset contiene 969 ciclos de fresado con señales de vibración y corriente.
Se carga directamente desde GitHub usando la URL raw del archivo CSV.

**Nota:** El archivo usa punto y coma (`;`) como separador y coma (`,`) como decimal, por eso se especifican los parámetros `sep` y `decimal`.

In [18]:
import pandas as pd

url = 'https://raw.githubusercontent.com/Coobaltoo13/milling-tool-life-fusion/main/data/raw/FeatureAndMetadata_Milling.csv'
# Load data using the second line of the CSV as headers (0-indexed header=1)
df = pd.read_csv(url, sep=';', decimal=',', header=1)

print(df.shape)
print(df.columns.tolist())
print(df.head())

(968, 131)
['Accelerometer - Spindle +Y - min', 'Accelerometer - Spindle -Z - min', 'Accelerometer - Spindle -X - min', 'Accelerometer - X Driving axle +Z - min', 'Accelerometer - X Driving axle -X - min', 'Accelerometer - Y Driving axle +Z - min', 'Accelerometer - Y Driving axle +Y - min', 'Accelerometer - Y Driving axle -X - min', 'Current - Spindle L1 - min', 'Current - Spindle L2 - min', 'Current - Spindle L3 - min', 'Current - Driving axle X L1 - min', 'Current - Driving axle X L2 - min', 'Current - Driving axle X L3 - min', 'Current - Driving axle Y L1 - min', 'Current - Driving axle Y L2 - min', 'Current - Driving axle Y L3 - min', 'Current - Driving axle Z L1 - min', 'Current - Driving axle Z L2 - min', 'Current - Driving axle Z L3 - min', 'Accelerometer - Spindle +Y - max', 'Accelerometer - Spindle -Z - max', 'Accelerometer - Spindle -X - max', 'Accelerometer - X Driving axle +Z - max', 'Accelerometer - X Driving axle -X - max', 'Accelerometer - Y Driving axle +Z - max', 'Acce

2. Resultados de la carga
El dataset tiene **968 filas** (ciclos de fresado) y **131 columnas** (mediciones).

Las columnas se dividen en:
- **Señales de sensores:** vibración (acelerómetros) y corriente
(husillo y ejes)
- **Metadatos:** número de ciclo, herramienta, dureza, etc.
- **Etiqueta:** `CycleToFailureNormalized` (vida útil normalizada)

In [17]:
# Ver la etiqueta (variable objetivo)
print("=" * 50)
print("VARIABLE OBJETIVO: CycleToFailureNormalized")
print("=" * 50)
print(df['CycleToFailureNormalized'].describe())
print(f"\nValores faltantes: {df['CycleToFailureNormalized'].isnull().sum()}")
print(f"Rango: {df['CycleToFailureNormalized'].min():.4f} a {df['CycleToFailureNormalized'].max():.4f}")

VARIABLE OBJETIVO: CycleToFailureNormalized
count    968.000000
mean       0.499025
std        0.291785
min        0.000000
25%        0.245845
50%        0.500000
75%        0.750000
max        1.000000
Name: CycleToFailureNormalized, dtype: float64

Valores faltantes: 0
Rango: 0.0000 a 1.0000


3. Conclusion

El dataset se cargó correctamente. Tiene 968 ciclos con señales de vibración y corriente. La etiqueta `CycleToFailureNormalized` es la variable a predecir.

**Siguiente paso:** Implementar un modelo baseline MLP + LSTM para predecir la vida útil.

In [19]:
print("Forma:", df.shape)
print("Etiqueta count:", df['CycleToFailureNormalized'].count())

Forma: (968, 131)
Etiqueta count: 968


------------------------------

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras import layers, models

print("TensorFlow:", tf.__version__)

TensorFlow: 2.20.0


In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras import layers, models

# Cargar
url = 'https://raw.githubusercontent.com/Coobaltoo13/milling-tool-life-fusion/main/data/raw/FeatureAndMetadata_Milling.csv'
df = pd.read_csv(url, sep=';', header=1, decimal=',')

# Strip whitespace from column names to ensure exact matches
df.columns = df.columns.str.strip()

# Identificar columnas no-feature (metadata y target) con los nombres correctos del DataFrame
non_feature_columns = ['FileName', 'TollIndex', 'HardnessMean', 'NumberOfCycle', 'CycleToFailure', 'ToolHolderLength', 'MillingToolType', 'ADOC', 'RDOC', 'SampleIndex']
target_column = 'CycleToFailureNormalized'

# Separate features (señales) and target (etiqueta)
# Ensure target_column is not accidentally included in non_feature_columns for dropping from X

# First, identify columns for X (features)
# Drop all non-feature columns, including the target column, from a copy of df to create X
X = df.drop(columns=non_feature_columns + [target_column]).apply(pd.to_numeric, errors='coerce')

# Then, extract y (target)
y = pd.to_numeric(df[target_column], errors='coerce')

# Eliminar filas con NaN en X o y
mask = ~(X.isnull().any(axis=1) | y.isnull())
X = X[mask].values
y = y[mask].values

print("X (señales):", X.shape)
print("y (etiqueta):", y.shape)

X (señales): (968, 120)
y (etiqueta): (968,)


In [34]:
#Normalizar
scaler = StandardScaler()

print("Shape of X before scaling:", X.shape)
print("Content of X (first 5 rows) before scaling:\n", X[:5])
X_scaled = scaler.fit_transform(X)

print("Datos normalizados")
print("Media:", X_scaled.mean().round(4))
print("Desviación:", X_scaled.std().round(4))

Shape of X before scaling: (968, 120)
Content of X (first 5 rows) before scaling:
 [[-2.06200000e+04 -2.00400000e+03 -2.09700000e+03 -1.93000000e+02
  -1.92000000e+02 -1.56000000e+02 -1.15000000e+02 -1.68000000e+02
   2.19600000e+03  2.21100000e+03  2.80500000e+03  2.96000000e+02
   3.10000000e+02  3.14000000e+02  2.61000000e+02  2.86000000e+02
   3.33000000e+02  3.08000000e+02  3.19000000e+02  3.60000000e+02
   2.00500000e+03  7.83000000e+02  1.97800000e+03  1.59000000e+02
   1.73000000e+02  1.17000000e+02  7.40000000e+01  1.63000000e+02
   3.57268000e+05  3.59996000e+05  3.60304000e+05  1.28540000e+04
   1.32970000e+04  1.29940000e+04  1.48570000e+04  1.59750000e+04
   1.52200000e+04  3.06960000e+04  2.94090000e+04  2.70590000e+04
   9.44601607e-01  2.42870650e+01  1.01797020e+01 -1.16455768e+01
   1.85618121e+01  2.00326062e+01 -1.24511221e+00  2.12875516e+01
   1.46717372e+05  1.47069346e+05  1.46761477e+05  5.10738545e+03
   5.21378203e+03  5.17752412e+03  4.42250318e+03  4.569157

In [35]:
# LSTM necesita forma (muestras, pasos_tiempo, características)
# Usamos 1 paso de tiempo con 121 características
X_lstm = X_scaled.reshape((X_scaled.shape[0], 1, X_scaled.shape[1]))

print("Forma para LSTM:", X_lstm.shape)

Forma para LSTM: (968, 1, 120)


In [36]:
#Dividir train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_lstm, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (774, 1, 120)
Test: (194, 1, 120)


In [37]:
# Obtener el número real de características
input_dim = X_scaled.shape[1]

print("Número de características:", input_dim)

model = models.Sequential([
    layers.Input(shape=(1, input_dim)),

    # MLP inicial
    layers.TimeDistributed(layers.Dense(64, activation='relu')),

    # LSTM para temporalidad
    layers.LSTM(32, return_sequences=False),

    # MLP final
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='linear')
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

Número de características: 120


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed                │ (None, 1, 64)          │         7,744 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,249 (83.00 KB)

 Trainable params: 21,249 (83.00 KB)

 Non-trainable params: 0 (0.00 B)

In [38]:
#Entrenar
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

Epoch 1/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - loss: 0.1509 - mae: 0.3060 - val_loss: 0.0556 - val_mae: 0.1857
Epoch 2/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0481 - mae: 0.1786 - val_loss: 0.0368 - val_mae: 0.1482
Epoch 3/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0316 - mae: 0.1417 - val_loss: 0.0288 - val_mae: 0.1293
Epoch 4/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0239 - mae: 0.1220 - val_loss: 0.0247 - val_mae: 0.1173
Epoch 5/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0194 - mae: 0.1078 - val_loss: 0.0218 - val_mae: 0.1060
Epoch 6/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0162 - mae: 0.0980 - val_loss: 0.0195 - val_mae: 0.1011
Epoch 7/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0135 - mae: 0.0883 - val_loss: 0.0166 - val_mae: 0.0903
Epoch 8/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0114 - mae: 0.0810 - val_loss: 0.0155 - val_mae: 0.0876
Epoch 9/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0104 - mae:

NOTA:

loss sigue bajando (0.0004 al final) → el modelo se está memorizando los datos de entrenamiento

val_loss se estancó en ~0.011 → el modelo no mejora en datos que no ha visto

Esto se llama overfitting. El modelo aprendió de memoria los 774 ciclos de entrenamiento, pero no aprendió a generalizar para ciclos nuevos.

¿Es malo? No, es normal en un baseline. Significa que el modelo es demasiado complejo para la cantidad de datos que tenemos, o que necesita regularización.

QUE SIGNIFICA ESTO:

El Escenario 1 ya está listo. Baseline funcionando.

El overfitting es esperado. Por eso importante comparar con el autoencoder: para ver si limpiando las señales se reduce el overfitting.


In [40]:
#Evaluar en el conjunto de prueba
y_pred = model.predict(X_test).flatten()

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n" + "=" * 40)
print("RESULTADOS - ESCENARIO 1 (MLP + LSTM)")
print("=" * 40)
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R²:   {r2:.4f}")

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step

RESULTADOS - ESCENARIO 1 (MLP + LSTM)
RMSE: 0.1116
MAE:  0.0761
R²:   0.8572


El modelo predice la vida útil de la herramienta con un error promedio de ~7.6% (en la escala normalizada 0-1). Bueno para baseline?